# Kubernetes Basics for LLM Inference

**Stage 3 - Production | Notebook 3**

## Learning Objectives
- Understand K8s primitives for ML: Pod, Deployment, Service, ConfigMap, Secret, HPA
- Deploy vLLM on K8s with GPU scheduling
- Set up monitoring with Prometheus, Grafana, and DCGM Exporter
- Configure log aggregation with Loki + Promtail
- Run local K8s with minikube/kind for testing

In [ ]:
!pip install kubernetes

## 1. K8s Concepts for ML Engineers

| Resource | Purpose | ML-Specific Example |
|----------|---------|---------------------|
| **Pod** | Smallest deployable unit | One vLLM container + metrics sidecar |
| **Deployment** | Manages replica Pods | 3 vLLM replicas for HA |
| **Service** | Stable network endpoint | ClusterIP for internal API, LoadBalancer for external |
| **ConfigMap** | Non-secret configuration | Model path, max-batch-size, log level |
| **Secret** | Sensitive data | API keys, HuggingFace token, TLS certs |
| **PersistentVolumeClaim** | Storage request | Model weights on shared storage |
| **HPA** | Auto-scaling | Scale vLLM pods based on QPS or GPU utilization |
| **DaemonSet** | One pod per node | DCGM Exporter for GPU metrics |

## 2. GPU Node Setup: nvidia-device-plugin

Every K8s GPU node needs the NVIDIA device plugin to expose GPUs as schedulable resources.

In [ ]:
%%writefile nvidia-device-plugin.yml
# Deployed as a DaemonSet — one pod per GPU node
# This registers nvidia.com/gpu as a schedulable resource
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: nvidia-device-plugin-daemonset
  namespace: kube-system
spec:
  selector:
    matchLabels:
      name: nvidia-device-plugin-ds
  template:
    metadata:
      labels:
        name: nvidia-device-plugin-ds
    spec:
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      containers:
        - name: nvidia-device-plugin-ctr
          image: nvcr.io/nvidia/k8s-device-plugin:v0.15.0
          env:
            - name: FAIL_ON_INIT_ERROR
              value: "false"
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          volumeMounts:
            - name: device-plugin
              mountPath: /var/lib/kubelet/device-plugins
      volumes:
        - name: device-plugin
          hostPath:
            path: /var/lib/kubelet/device-plugins
---
# Time-slicing config for GPU sharing (optional)
apiVersion: v1
kind: ConfigMap
metadata:
  name: time-slicing-config
  namespace: kube-system
data:
  any: |-
    version: v1
    flags:
      migStrategy: none
    sharing:
      timeSlicing:
        resources:
          - name: nvidia.com/gpu
            replicas: 4   # 4 slices per GPU

# Apply: kubectl apply -f nvidia-device-plugin.yml

## 3. vLLM Deployment YAML with GPU Limits

In [ ]:
%%writefile vllm-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-inference
  namespace: llm
  labels:
    app: vllm
spec:
  replicas: 2
  selector:
    matchLabels:
      app: vllm
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0      # Zero-downtime for inference
  template:
    metadata:
      labels:
        app: vllm
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "8000"
        prometheus.io/path: "/metrics"
    spec:
      # ── GPU Scheduling ──
      nodeSelector:
        accelerator: nvidia-a100   # Pin to GPU nodes
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule

      containers:
        - name: vllm
          image: vllm/vllm-openai:v0.6.3.post1
          command:
            - python3
            - -m
            - vllm.entrypoints.openai.api_server
          args:
            - "--host"
            - "0.0.0.0"
            - "--port"
            - "8000"
            - "--model"
            - "$(MODEL_PATH)"
            - "--max-model-len"
            - "$(MAX_MODEL_LEN)"
            - "--gpu-memory-utilization"
            - "$(GPU_MEM_UTIL)"
          env:
            - name: MODEL_PATH
              valueFrom:
                configMapKeyRef:
                  name: vllm-config
                  key: model_path
            - name: MAX_MODEL_LEN
              valueFrom:
                configMapKeyRef:
                  name: vllm-config
                  key: max_model_len
            - name: GPU_MEM_UTIL
              valueFrom:
                configMapKeyRef:
                  name: vllm-config
                  key: gpu_memory_utilization
            - name: HF_TOKEN
              valueFrom:
                secretKeyRef:
                  name: hf-credentials
                  key: token
          ports:
            - containerPort: 8000
              name: http
          resources:
            requests:
              nvidia.com/gpu: 1          # Request 1 GPU
              memory: "32Gi"
              cpu: "8"
            limits:
              nvidia.com/gpu: 1          # Limit 1 GPU
              memory: "64Gi"
              cpu: "16"
          volumeMounts:
            - name: models
              mountPath: /models
              readOnly: true
            - name: shm
              mountPath: /dev/shm      # Shared memory for NCCL
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 120
            periodSeconds: 30
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 60
            periodSeconds: 10

      volumes:
        - name: models
          persistentVolumeClaim:
            claimName: model-weights-pvc
        - name: shm
          emptyDir:
            medium: Memory
            sizeLimit: 8Gi

print("Apply: kubectl apply -f vllm-deployment.yaml")

## 4. Service + Ingress for API Exposure

In [ ]:
%%writefile vllm-service.yaml
# ── ClusterIP Service for internal traffic ──
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
  namespace: llm
spec:
  type: ClusterIP
  selector:
    app: vllm
  ports:
    - port: 8000
      targetPort: 8000
      protocol: TCP
      name: http
  sessionAffinity: ClientIP    # Sticky sessions for streaming
  sessionAffinityConfig:
    clientIP:
      timeoutSeconds: 3600
---
# ── Ingress for external access ──
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: vllm-ingress
  namespace: llm
  annotations:
    nginx.ingress.kubernetes.io/proxy-read-timeout: "3600"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "3600"
    nginx.ingress.kubernetes.io/proxy-body-size: "10m"
    cert-manager.io/cluster-issuer: letsencrypt-prod
spec:
  ingressClassName: nginx
  tls:
    - hosts:
        - api.mycompany.com
      secretName: vllm-tls
  rules:
    - host: api.mycompany.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: vllm-service
                port:
                  number: 8000

print("Apply: kubectl apply -f vllm-service.yaml")

## 5. ConfigMap for Model Config, Secret for API Keys

In [ ]:
%%writefile vllm-configmap.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: vllm-config
  namespace: llm
data:
  model_path: "/models/Meta-Llama-3-70B-Instruct"
  max_model_len: "32768"
  gpu_memory_utilization: "0.92"
  max_num_seqs: "256"
  max_num_batched_tokens: "8192"
  log_level: "INFO"
---
apiVersion: v1
kind: Secret
metadata:
  name: hf-credentials
  namespace: llm
type: Opaque
stringData:                   # Use stringData to avoid base64 encoding manually
  token: "hf_your_token_here"
---
apiVersion: v1
kind: Secret
metadata:
  name: api-keys
  namespace: llm
type: Opaque
stringData:
  vllm-api-key: "sk-prod-key-change-me"

print("⚠️  Never commit plaintext secrets. Use kubeseal or external-secrets-operator in production.")

## 6. HPA (Horizontal Pod Autoscaler) QPS-Based Scaling

In [ ]:
%%writefile vllm-hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-hpa
  namespace: llm
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm-inference
  minReplicas: 2
  maxReplicas: 10
  metrics:
    # Scale based on requests per second (custom metric from Prometheus)
    - type: Pods
      pods:
        metric:
          name: http_requests_per_second
        target:
          type: AverageValue
          averageValue: "50"       # 50 QPS per pod triggers scale-up
    # Scale based on GPU utilization from DCGM
    - type: Pods
      pods:
        metric:
          name: DCGM_FI_DEV_GPU_UTIL
        target:
          type: AverageValue
          averageValue: "70"       # 70% GPU util triggers scale-up
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300  # Wait 5 min before scaling down
      policies:
        - type: Percent
          value: 50
          periodSeconds: 60
    scaleUp:
      stabilizationWindowSeconds: 30    # Scale up quickly
      policies:
        - type: Percent
          value: 100
          periodSeconds: 30

print("Apply: kubectl apply -f vllm-hpa.yaml")
print("Check: kubectl get hpa -n llm --watch")

## 7. Prometheus + Grafana Monitoring Setup

In [ ]:
%%writefile prometheus-stack.sh
# Install kube-prometheus-stack (Prometheus + Grafana + AlertManager)
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
helm repo update

helm install prometheus prometheus-community/kube-prometheus-stack \
  --namespace monitoring \
  --create-namespace \
  --set prometheus.prometheusSpec.serviceMonitorSelectorNilUsesHelmValues=false \
  --set grafana.adminPassword=admin123

# Port-forward Grafana
# kubectl port-forward -n monitoring svc/prometheus-grafana 3000:80
# Open http://localhost:3000 (admin / admin123)

## 8. DCGM Exporter for GPU Metrics

In [ ]:
%%writefile dcgm-exporter.yaml
# NVIDIA DCGM Exporter — exposes GPU metrics to Prometheus
helm repo add nvidia https://helm.ngc.nvidia.com/nvidia
helm repo update

# Install DCGM Exporter
helm install dcgm-exporter nvidia/dcgm-exporter \
  --namespace monitoring \
  --set serviceMonitor.enabled=true \
  --set serviceMonitor.additionalLabels.release=prometheus

# Key GPU metrics exposed:
# DCGM_FI_DEV_GPU_UTIL          — GPU utilization (%)
# DCGM_FI_DEV_MEM_COPY_UTIL     — Memory bandwidth utilization
# DCGM_FI_DEV_FB_USED           — Framebuffer memory used (MiB)
# DCGM_FI_DEV_FB_FREE           — Framebuffer memory free (MiB)
# DCGM_FI_DEV_POWER_USAGE       — Power draw (W)
# DCGM_FI_DEV_GPU_TEMP          — Temperature (C)
# DCGM_FI_DEV_SM_CLOCK          — Streaming Multiprocessor clock (MHz)

## 9. Loki + Promtail for Logs

In [ ]:
%%writefile loki-stack.sh
# Install Loki + Promtail for log aggregation
helm repo add grafana https://grafana.github.io/helm-charts
helm repo update

helm install loki grafana/loki-stack \
  --namespace monitoring \
  --set loki.persistence.enabled=true \
  --set loki.persistence.size=50Gi \
  --set promtail.tolerations[0].key=nvidia.com/gpu \
  --set promtail.tolerations[0].operator=Exists \
  --set promtail.tolerations[0].effect=NoSchedule

# In Grafana, add Loki as a data source:
# URL: http://loki.monitoring.svc.cluster.local:3100

# Example LogQL queries:
# {app="vllm", namespace="llm"} |= "ERROR"
# {app="vllm"} | json | line_format "{{.message}}"
# rate({app="vllm"} |~ "request_id" [5m])

## 10. kubectl Cheatsheet

Essential commands for managing LLM workloads on K8s.

In [ ]:
print("""
| Command | Description |
|---------|-------------|
| `kubectl get pods -n llm -w` | Watch pod status |
| `kubectl describe pod <name> -n llm` | Pod details (events, resources) |
| `kubectl logs -f <pod> -n llm` | Stream pod logs |
| `kubectl logs <pod> -n llm --previous` | Logs of crashed container |
| `kubectl exec -it <pod> -n llm -- nvidia-smi` | GPU status inside pod |
| `kubectl exec -it <pod> -n llm -- /bin/bash` | Shell inside pod |
| `kubectl get events -n llm --sort-by='.lastTimestamp'` | Recent namespace events |
| `kubectl top pods -n llm` | Pod resource usage |
| `kubectl top nodes` | Node resource usage |
| `kubectl rollout restart deployment/vllm-inference -n llm` | Rolling restart |
| `kubectl rollout history deployment/vllm-inference -n llm` | Deployment history |
| `kubectl rollout undo deployment/vllm-inference -n llm` | Rollback |
| `kubectl scale deployment/vllm-inference --replicas=5 -n llm` | Manual scale |
| `kubectl cordon <node>` | Mark node as unschedulable |
| `kubectl drain <node> --ignore-daemonsets` | Evacuate node for maintenance |
| `kubectl get pvc -n llm` | Check persistent volume claims |
| `kubectl get svc,endpoints -n llm` | Services and their endpoints |
| `kubectl apply -f manifest.yaml --dry-run=client` | Validate manifest |
| `kubectl diff -f manifest.yaml` | Show changes before applying |
""")

## 11. Local K8s with minikube/kind

For local testing before deploying to production clusters.

In [ ]:
# ── Option A: minikube (GPU support experimental) ──
# brew install minikube
# minikube start --cpus=8 --memory=32768 --driver=docker
# minikube addons enable ingress
# minikube addons enable metrics-server

# ── Option B: kind (Kubernetes in Docker) ──
# brew install kind
# kind create cluster --name llm-test --config kind-config.yaml

kind_config = """
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
  - role: control-plane
  - role: worker
    kubeadmConfigPatches:
      - |
        kind: JoinConfiguration
        nodeRegistration:
          kubeletExtraArgs:
            system-reserved: memory=8Gi
    extraMounts:
      - hostPath: /mnt/models
        containerPath: /models
"""
print(kind_config)

print("\n# Set kubectl context after creating cluster:")
print("kubectl cluster-info --context kind-llm-test")

## 12. Exercise: Complete K8s Manifest for Full Stack

**TODO:** Build a complete K8s manifest that includes:
1. A Deployment for vLLM with:
   - GPU resource requests and limits
   - Liveness and readiness probes
   - ConfigMap-based configuration
2. A Service of type ClusterIP with session affinity
3. An Ingress with TLS and long timeouts (for streaming)
4. An HPA that scales on both QPS and GPU utilization
5. A PodDisruptionBudget (minAvailable: 1)
6. A NetworkPolicy that restricts ingress traffic to the API gateway only

In [ ]:
%%writefile exercise-full-stack.yaml
# TODO: Complete this manifest with all 6 components listed above

# 1. Deployment (fill in the blanks)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-inference
  namespace: llm
spec:
  replicas: __FILL__
  selector:
    matchLabels:
      app: vllm
  template:
    spec:
      containers:
        - name: vllm
          image: __FILL__
          resources:
            requests:
              nvidia.com/gpu: __FILL__
            limits:
              nvidia.com/gpu: __FILL__
          livenessProbe:
            # YOUR CODE HERE
          readinessProbe:
            # YOUR CODE HERE
          env:
            - name: MODEL_PATH
              valueFrom:
                configMapKeyRef:
                  name: __FILL__
                  key: model_path
---
# 2. Service
# YOUR CODE HERE
---
# 3. Ingress
# YOUR CODE HERE
---
# 4. HPA
# YOUR CODE HERE
---
# 5. PodDisruptionBudget
# YOUR CODE HERE
---
# 6. NetworkPolicy
# YOUR CODE HERE

print("Fill in all __FILL__ placeholders and add the missing resource definitions.")

## Summary

In this notebook, you learned:
- **K8s primitives** (Pod, Deployment, Service, ConfigMap, Secret) for LLM workloads
- **GPU scheduling** with nvidia-device-plugin and resource limits
- **Service and Ingress** for internal and external API exposure
- **ConfigMap and Secret** for separating config from code
- **HPA** for QPS and GPU-based auto-scaling
- **Prometheus + Grafana + DCGM Exporter** for GPU monitoring
- **Loki + Promtail** for log aggregation
- **kubectl cheatsheet** for daily operations
- **Local K8s** with minikube/kind for local testing
- **Full-stack exercise** to reinforce all concepts

**Next:** CI/CD pipelines and GitOps for LLM deployments.